In [4]:
"""模型对比测试：retrieval_query_generation 模板

使用 retrieval_query_generation.md 模板，用同一组填充内容（description + prompt）
在 MODEL_ID_1 与 MODEL_ID_2 下分别生成检索 query，并对比输出与用时。
"""

import os
import sys
import time

# 确保项目根目录在 sys.path 中
PROJECT_ROOT = r"e:\yusp-202506-Work\yusys_ai"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.utils.model_factory import ChatModelFactory
from src.agents.content_generate_agent.utils import apply_prompt_template

# ========== 超参数：两个对比模型 ==========
MODEL_ID_1 = "model_bailian_80b"
MODEL_ID_2 = "model_4b_instruct"

# ========== 多组模板填充内容（多轮对比） ==========
# 每项为 {"description": ..., "prompt": ...}，与 retrieval_query_generation.md 中的 {{ description }}、{{ prompt }} 对应
TEMPLATE_CASES = [
    {
        "description": (
            "申请人的基础设施建设业务有哪些？基础设施板块中PPP项目的数量、投资与回款情况如何？"
            "回报机制以何种方式为主？是否存在回款周期长和运营不确定性风险？"
        ),
        "prompt": "请聚焦最近三年的数据，仅统计已完工项目；若无则说明暂无。",
    },
    {
        "description": "授信初审报告中的审查人是谁？",
        "prompt": "",
    },
    {
        "description": "分行初审意见及联系方式",
        "prompt": "需包含初审结论与联系人电话。",
    },
    {
        "description": "近年来公司新项目承揽情况：新签合同金额、政国企类占比、5亿元及以上合同金额占比，按三年一期整理。",
        "prompt": "表头为项目、2022年、2023年、2024年、2025.1-3，只填数据。",
    },
    {
        "description": "发行人报告期内是否存在重大违法违规行为？是否受到监管处罚？",
        "prompt": "仅限行政处罚与重大诉讼，不含一般合规说明。",
    },
]

print("当前配置：")
print("  MODEL_ID_1 =", MODEL_ID_1)
print("  MODEL_ID_2 =", MODEL_ID_2)
print("  模板 = retrieval_query_generation.md")
print("  测试用例数 =", len(TEMPLATE_CASES))
for i, c in enumerate(TEMPLATE_CASES):
    d = c["description"][:40] + "..." if len(c["description"]) > 40 else c["description"]
    p = c["prompt"] or "(空)"
    print(f"  [{i+1}] description: {d}  | prompt: {p}")


当前配置：
  MODEL_ID_1 = model_bailian_80b
  MODEL_ID_2 = model_4b_instruct
  模板 = retrieval_query_generation.md
  测试用例数 = 5
  [1] description: 申请人的基础设施建设业务有哪些？基础设施板块中PPP项目的数量、投资与回款情况如...  | prompt: 请聚焦最近三年的数据，仅统计已完工项目；若无则说明暂无。
  [2] description: 授信初审报告中的审查人是谁？  | prompt: (空)
  [3] description: 分行初审意见及联系方式  | prompt: 需包含初审结论与联系人电话。
  [4] description: 近年来公司新项目承揽情况：新签合同金额、政国企类占比、5亿元及以上合同金额占比，...  | prompt: 表头为项目、2022年、2023年、2024年、2025.1-3，只填数据。
  [5] description: 发行人报告期内是否存在重大违法违规行为？是否受到监管处罚？  | prompt: 仅限行政处罚与重大诉讼，不含一般合规说明。


In [15]:
config_path = os.path.join(PROJECT_ROOT, "config", "model_factory_config.json")
factory = ChatModelFactory(config_path=config_path)
model1 = factory.create_langchain_model(model_id=MODEL_ID_1)
model2 = factory.create_langchain_model(model_id=MODEL_ID_2)

# 多轮对比：对每组 (description, prompt) 渲染模板、调用两模型、记录输出与用时
COMPARE_RESULTS = []

for idx, case in enumerate(TEMPLATE_CASES):
    full_prompt = apply_prompt_template(
        "retrieval_query_generation",
        {"description": case["description"], "prompt": case["prompt"] or ""},
    )
    print(f"\n{'='*60}")
    print(f"【用例 {idx+1}/{len(TEMPLATE_CASES)}】")
    print(f"  description: {case['description'][:50]}{'...' if len(case['description'])>50 else ''}")
    print(f"  prompt: {case['prompt'] or '(空)'}")

    t0 = time.perf_counter()
    resp1 = model1.invoke(full_prompt)
    elapsed1 = time.perf_counter() - t0
    out1 = (resp1.content if hasattr(resp1, "content") else str(resp1)).strip()

    t0 = time.perf_counter()
    resp2 = model2.invoke(full_prompt)
    elapsed2 = time.perf_counter() - t0
    out2 = (resp2.content if hasattr(resp2, "content") else str(resp2)).strip()

    faster = MODEL_ID_1 if elapsed1 < elapsed2 else MODEL_ID_2
    COMPARE_RESULTS.append({
        "idx": idx + 1,
        "description": case["description"],
        "prompt": case["prompt"],
        "out1": out1,
        "out2": out2,
        "elapsed1": elapsed1,
        "elapsed2": elapsed2,
        "faster": faster,
    })

    print(f"  --- {MODEL_ID_1} [用时 {elapsed1:.2f}s] ---")
    print(f"      {out1}")
    print(f"  --- {MODEL_ID_2} [用时 {elapsed2:.2f}s] ---")
    print(f"      {out2}")
    faster_time = elapsed1 if faster == MODEL_ID_1 else elapsed2
    slower_time = elapsed2 if faster == MODEL_ID_1 else elapsed1
    ratio = faster_time / slower_time if slower_time > 0 else 1.0
    print(f"  更快: {faster}（{faster_time:.2f}/{slower_time:.2f}={ratio*100:.0f}%）")

# ----- 汇总表：多次对比结果 -----
print("\n" + "="*60)
print("========== 多次对比结果汇总 ==========")
print(f"{'用例':<6} {MODEL_ID_1} 用时(s)  {MODEL_ID_2} 用时(s)  更快")
print("-"*60)
for r in COMPARE_RESULTS:
    print(f"  {r['idx']}     {r['elapsed1']:.2f}       {r['elapsed2']:.2f}       {r['faster']}")
print("-"*60)
total1 = sum(r["elapsed1"] for r in COMPARE_RESULTS)
total2 = sum(r["elapsed2"] for r in COMPARE_RESULTS)
print(f"  合计   {total1:.2f}       {total2:.2f}       {MODEL_ID_1 if total1 < total2 else MODEL_ID_2}")
print("\n各用例检索 query 输出已记录在 COMPARE_RESULTS 中（out1 / out2）。")


2026-02-25 17:20:31.330 | INFO     | src.utils.model_factory:create_langchain_model:54 - Using default base_url[http://103.154.31.78:20001/compatible-mode/v1] and api_key[****] from environment variables for model model_bailian_80b



【用例 1/5】
  description: 申请人的基础设施建设业务有哪些？基础设施板块中PPP项目的数量、投资与回款情况如何？回报机制以何种方...
  prompt: 请聚焦最近三年的数据，仅统计已完工项目；若无则说明暂无。
  --- model_bailian_80b [用时 0.91s] ---
      近三年已完工基础设施PPP项目数量、投资、回款情况及回报机制，是否存在回款周期长和运营不确定性风险
  --- model_4b_instruct [用时 0.32s] ---
      近三年已完工基础设施PPP项目投资回款及回报机制风险
  更快: model_4b_instruct（0.32/0.91=35%）

【用例 2/5】
  description: 授信初审报告中的审查人是谁？
  prompt: (空)
  --- model_bailian_80b [用时 0.74s] ---
      授信初审报告中的审查人是谁
  --- model_4b_instruct [用时 0.20s] ---
      授信初审报告的审查人是谁
  更快: model_4b_instruct（0.20/0.74=27%）

【用例 3/5】
  description: 分行初审意见及联系方式
  prompt: 需包含初审结论与联系人电话。
  --- model_bailian_80b [用时 0.53s] ---
      分行初审意见及联系人电话
  --- model_4b_instruct [用时 0.20s] ---
      分行初审意见及联系人电话
  更快: model_4b_instruct（0.20/0.53=38%）

【用例 4/5】
  description: 近年来公司新项目承揽情况：新签合同金额、政国企类占比、5亿元及以上合同金额占比，按三年一期整理。
  prompt: 表头为项目、2022年、2023年、2024年、2025.1-3，只填数据。
  --- model_bailian_80b [用时 0.88s] ---
      近三年及2025年一季度新签合同金额、政国企占比、5亿以上合同占比数据
  --- model_4b_instruct [用时 0.39s] ---
      

In [ ]:
"""多模板综合测试：遍历 content_generate_agent/prompts 下的所有模板

说明：
- 使用同一对模型 MODEL_ID_1 / MODEL_ID_2
- 对每个 prompt template 准备若干组示例 state（变量），通过 apply_prompt_template 渲染
- 记录每个模板、每个用例下两模型的输出和用时，便于统一对比
"""

from pprint import pprint
from langchain_core.messages import SystemMessage, HumanMessage

# 重新构造工厂与模型（ChatModelFactory 是单例，多次初始化无伤大雅）
config_path = os.path.join(PROJECT_ROOT, "config", "model_factory_config.json")
factory = ChatModelFactory(config_path=config_path)
model1 = factory.create_langchain_model(model_id=MODEL_ID_1)
model2 = factory.create_langchain_model(model_id=MODEL_ID_2)

# 一些通用示例片段，供不同模板复用
EXAMPLE_QUESTION_BASIC = "公司2023年的营业收入和净利润是多少？"
EXAMPLE_QUESTION_RISK = "公司报告期内面临的主要风险有哪些？"
EXAMPLE_CONTEXT_FINANCE = (
    "公司2023年实现营业收入77,345.13万元，同比增长4.79%，"
    "归属于母公司所有者的净利润为11,106.50万元，同比上升6.21%。"
)
EXAMPLE_CONTEXT_RISK = (
    "公司面临的主要风险包括市场需求波动风险、客户高度集中风险、"
    "以及原材料价格波动风险等。公司已通过签订长期合同和多元化客户策略进行应对。"
)

EXAMPLE_TABLE_HTML_SIMPLE = (
    "<table><tr><th>项目</th><th>2022年</th><th>2023年</th></tr>"
    "<tr><td>营业收入(万元)</td><td>70000</td><td>77345.13</td></tr>"
    "<tr><td>净利润(万元)</td><td>10000</td><td>11106.50</td></tr></table>"
)
EXAMPLE_TABLE_HTML_COMPLEX = (
    "<table><tr><td rowspan='2'></td><td rowspan='2'>2023年</td><td colspan='2'>2022年</td></tr>"
    "<tr><td>调整前</td><td>调整后</td></tr>"
    "<tr><td>营业收入(元)</td><td>773,451,288.74</td><td>738,068,573.03</td></tr>"
    "<tr><td>净利润(元)</td><td>111,065,035.62</td><td>104,536,221.66</td></tr></table>"
)

EXAMPLE_PIECES_CONTENT = (
    "doc_1: 公司2023年实现营业收入77,345.13万元，同比增长4.79%。\n"
    "doc_2: 公司2023年归属于母公司所有者的净利润为11,106.50万元，同比上升6.21%。\n"
    "doc_3: 公司主要风险包括市场需求波动和客户集中度较高等。"
)
EXAMPLE_CHAPTERS_TEXT = (
    "1. 公司基本情况：包括主营业务、股权结构等。\n"
    "2. 财务状况：披露了2020-2023年的收入、利润等关键指标。\n"
    "3. 风险因素：分析了市场风险、信用风险及流动性风险。"
)

# faithfulness_statement_verification / extraction 用的示例
EXAMPLE_FAITH_QUESTION = "公司2023年的营业收入和净利润是多少？"
EXAMPLE_FAITH_ANSWER = (
    "公司2023年实现营业收入77,345.13万元，较上年增长4.79%，"
    "归属于母公司所有者的净利润为11,106.50万元，同比上升6.21%。"
)
EXAMPLE_FAITH_CONTEXT = (
    "年报披露：公司2023年实现营业收入77,345.13万元，同比增长4.79%，"
    "归属于母公司所有者的净利润为11,106.50万元，同比上升6.21%。"
)
EXAMPLE_FAITH_STATEMENTS = [
    {"id": 1, "statement": "公司2023年实现营业收入77,345.13万元。"},
    {"id": 2, "statement": "公司2023年净利润同比上升6.21%。"},
]

# block / content 相关示例
EXAMPLE_SELECTED_CONTENT = (
    "文档片段1：公司2020-2023年营业收入分别为65,000.00、70,200.50、73,800.20、77,345.13万元，"
    "归属于母公司所有者的净利润分别为8,900.00、9,750.30、10,320.40、11,106.50万元。\n"
    "文档片段2：分析了公司所在行业的市场竞争格局和主要风险因素，包括主要竞争者、市占率变化及行业景气度等。"
)

# ====== 为每个 template 准备若干组示例 state ======
TEMPLATE_TEST_CASES = {
    # 已在前面单独测试过的 retrieval_query_generation，这里复用 TEMPLATE_CASES
    "retrieval_query_generation": TEMPLATE_CASES,

    "content_generation_merge_windows": [
        {
            "question": EXAMPLE_QUESTION_BASIC,
            "window_answers": (
                "【窗口1答案】公司2023年实现营业收入77,345.13万元，同比增长4.79%。[1]\n"
                "【窗口2答案】归属于母公司所有者的净利润为11,106.50万元，同比上升6.21%。[2]"
            ),
        },
        {
            "question": EXAMPLE_QUESTION_RISK,
            "window_answers": (
                "【窗口1答案】公司面临市场需求波动风险和客户集中度较高风险。[3]\n"
                "【窗口2答案】公司还面临原材料价格波动风险。[4]"
            ),
        },
        {
            "question": "公司注册地址和法定代表人是谁？",
            "window_answers": (
                "【窗口1答案】公司注册地址：广东省东莞市厚街镇恒通路10号。[5]\n"
                "【窗口2答案】法定代表人：张三。[6]"
            ),
        },
        {
            "question": "报告期内研发投入及专利情况如何？",
            "window_answers": (
                "【窗口1答案】报告期内研发费用75,165万元，占营业收入比例约9.7%。[7]\n"
                "【窗口2答案】公司拥有发明专利31项、实用新型99项、外观设计6项。[8]"
            ),
        },
        {
            "question": "募集资金用途及使用进度？",
            "window_answers": (
                "【窗口1答案】本期债券募集资金拟用于补充流动资金及偿还债务。[9]\n"
                "【窗口2答案】截至报告期末，募集资金已使用约60%，剩余部分按计划投入。[10]"
            ),
        },
    ],

    "content_generation_system": [
        {"simple_answer": False},
        {"simple_answer": True},
        {"simple_answer": False},
        {"simple_answer": True},
    ],

    "content_generation_human": [
        {
            "question": EXAMPLE_QUESTION_BASIC,
            "context_text": EXAMPLE_CONTEXT_FINANCE,
            "extra_instruction": "回答控制在200字以内，需给出具体数值并简要点评。",
        },
        {
            "question": EXAMPLE_QUESTION_RISK,
            "context_text": EXAMPLE_CONTEXT_RISK,
            "extra_instruction": "按条列方式列出主要风险，每条不超过30字。",
        },
        {
            "question": "公司注册地址在哪里？",
            "context_text": "公司注册地为广东省东莞市厚街镇恒通路10号，经营范围为软件开发与技术服务。",
            "extra_instruction": "仅回答地址，不要扩展。",
        },
        {
            "question": "报告期内研发投入金额及占比？",
            "context_text": "公司2023年研发费用75,165万元，占营业收入比例约9.7%，较上年提高0.5个百分点。",
            "extra_instruction": "给出数值并简要说明趋势。",
        },
        {
            "question": "募集资金的主要用途是什么？",
            "context_text": "本期债券募集资金拟用于补充公司流动资金及偿还到期债务，其中70%用于补流，30%用于偿债。",
            "extra_instruction": "分条列出用途及比例。",
        },
    ],

    "table_segmentation_with_html": [
        {"table_html": EXAMPLE_TABLE_HTML_COMPLEX},
    ],

    "table_fillin": [
        {
            "extra_instruction": "仅补全缺失的数值，不要修改已有数据。",
            "context_text": EXAMPLE_CONTEXT_FINANCE,
            "table_header": "主营业务收入及净利润情况表",
            "table_html": EXAMPLE_TABLE_HTML_SIMPLE,
        }
    ],

    "table_segmentation": [
        {"table_html": EXAMPLE_TABLE_HTML_COMPLEX},
    ],

    "summary_generation": [
        {"context_text": EXAMPLE_CONTEXT_FINANCE},
        {"context_text": EXAMPLE_CONTEXT_RISK},
        {
            "context_text": "公司注册地为广东省东莞市厚街镇恒通路10号，法定代表人张三，主营业务为软件开发与技术服务，报告期内经营稳定。",
        },
        {
            "context_text": "报告期内公司研发费用75,165万元，占营业收入约9.7%；拥有发明专利31项、实用新型99项、外观设计6项，多项专利已应用于生产。",
        },
        {
            "context_text": "本期债券募集资金拟用于补充流动资金及偿还债务，截至报告期末已使用约60%，剩余部分将按计划用于项目建设及偿债。",
        },
    ],

    "quality_score": [
        {
            "reference": "公司注册地址为广东省东莞市厚街镇恒通路10号。",
            "response": "公司注册地址在广东省东莞市厚街镇恒通路10号。",
        },
        {
            "reference": "公司共拥有专利权136项，其中发明专利31项，实用新型99项，外观设计6项。",
            "response": "公司拥有专利权136项，其中包括发明专利和实用新型。",
        },
    ],

    "planner": [
        {
            "question": EXAMPLE_QUESTION_BASIC,
            "content": EXAMPLE_SELECTED_CONTENT,
            "coder_output": "目前尚未进行任何数值汇总计算。",
        },
        {
            "question": EXAMPLE_QUESTION_RISK,
            "content": EXAMPLE_SELECTED_CONTENT,
            "coder_output": "",
        },
        {
            "question": "公司注册地址在哪里？",
            "content": "第一章介绍公司基本情况，其中载明注册地为广东省东莞市厚街镇恒通路10号。",
            "coder_output": "",
        },
        {
            "question": "研发费用占营业收入比例是多少？",
            "content": "财务章节披露研发费用75,165万元；营业收入773,451万元。",
            "coder_output": "已提取研发费用与营业收入数值，尚未计算占比。",
        },
        {
            "question": "募集资金已使用比例？",
            "content": "募集说明书载明：截至报告期末募集资金已使用约60%。",
            "coder_output": "已从原文摘录使用比例，无需计算。",
        },
    ],

    "planner_new": [
        {
            "question": EXAMPLE_QUESTION_BASIC,
            "content": EXAMPLE_SELECTED_CONTENT,
            "coder_output": "已提取出财务数据表格，但尚未计算同比增速。",
        },
        {
            "question": EXAMPLE_QUESTION_RISK,
            "content": EXAMPLE_SELECTED_CONTENT,
            "coder_output": "",
        },
        {
            "question": "公司注册地址和法定代表人？",
            "content": "公司注册地为广东省东莞市厚街镇恒通路10号，法定代表人为张三。",
            "coder_output": "",
        },
        {
            "question": "专利数量及类型分布？",
            "content": "公司拥有专利权136项，其中发明专利31项，实用新型99项，外观设计6项。",
            "coder_output": "已列出各类型数量，无需进一步计算。",
        },
        {
            "question": "募集资金用途及已使用比例？",
            "content": "募集资金用于补充流动资金及偿债；截至报告期末已使用约60%。",
            "coder_output": "已从原文摘录用途与比例。",
        },
    ],

    "faithfulness_statement_verification": [
        {
            "context_text": EXAMPLE_FAITH_CONTEXT,
            "statements": EXAMPLE_FAITH_STATEMENTS,
        }
    ],

    "helpfulness_score": [
        {
            "question": "公司2023年的营业收入是多少？",
            "response": "公司2023年实现营业收入77,345.13万元。",
        },
        {
            "question": "公司有哪些主要风险？",
            "response": "公司面临市场需求波动和客户集中度较高风险。",
        },
    ],

    "faithfulness_statement_extraction": [
        {
            "question": EXAMPLE_FAITH_QUESTION,
            "answer": EXAMPLE_FAITH_ANSWER,
        }
    ],

    "content_sufficiency": [
        {
            "question": EXAMPLE_QUESTION_BASIC,
            "selected_content": EXAMPLE_SELECTED_CONTENT,
        },
        {
            "question": EXAMPLE_QUESTION_RISK,
            "selected_content": "仅包含公司基本情况介绍，未涉及风险披露。",
        },
    ],

    "content_filter": [
        {
            "question": EXAMPLE_QUESTION_BASIC,
            "total_pieces": 3,
            "pieces_content": EXAMPLE_PIECES_CONTENT,
        },
        {
            "question": EXAMPLE_QUESTION_RISK,
            "total_pieces": 3,
            "pieces_content": EXAMPLE_PIECES_CONTENT,
        },
        {
            "question": "公司注册地址在哪里？",
            "total_pieces": 2,
            "pieces_content": "doc_1: 公司基本情况。公司注册地为广东省东莞市厚街镇恒通路10号。\ndoc_2: 经营范围为软件开发与技术服务。",
        },
        {
            "question": "研发费用及专利情况？",
            "total_pieces": 2,
            "pieces_content": "doc_1: 报告期内研发费用75,165万元，占营业收入约9.7%。\ndoc_2: 公司拥有发明专利31项、实用新型99项、外观设计6项。",
        },
        {
            "question": "募集资金用途与使用进度？",
            "total_pieces": 2,
            "pieces_content": "doc_1: 募集资金拟用于补充流动资金及偿还债务。\ndoc_2: 截至报告期末已使用约60%，剩余部分按计划投入。",
        },
    ],

    "chapter_selection": [
        {
            "question": EXAMPLE_QUESTION_BASIC,
            "chapters_text": EXAMPLE_CHAPTERS_TEXT,
        },
        {
            "question": EXAMPLE_QUESTION_RISK,
            "chapters_text": EXAMPLE_CHAPTERS_TEXT,
        },
        {
            "question": "公司注册地址和法定代表人？",
            "chapters_text": "1. 公司基本情况：注册地、法定代表人、经营范围。\n2. 管理层与治理结构。\n3. 经营情况分析。",
        },
        {
            "question": "研发投入与专利情况？",
            "chapters_text": "1. 公司概况。\n2. 研发与创新能力：研发费用、专利数量及应用。\n3. 风险因素。",
        },
        {
            "question": "募集资金使用情况？",
            "chapters_text": "1. 募集资金概况。\n2. 募集资金使用情况：用途、使用进度、变更情况。\n3. 募投项目效益。",
        },
    ],

    "block_selection_structured_simple": [
        {
            "block_index": 1,
            "total_pieces": 3,
            "pieces_content": EXAMPLE_PIECES_CONTENT,
            "question": EXAMPLE_QUESTION_BASIC,
        },
        {
            "block_index": 1,
            "total_pieces": 3,
            "pieces_content": EXAMPLE_PIECES_CONTENT,
            "question": EXAMPLE_QUESTION_RISK,
        },
        {
            "block_index": 2,
            "total_pieces": 2,
            "pieces_content": "1. 公司注册地为广东省东莞市厚街镇恒通路10号。\n2. 经营范围为软件开发与技术服务。",
            "question": "公司注册地址在哪里？",
        },
        {
            "block_index": 1,
            "total_pieces": 2,
            "pieces_content": "1. 研发费用75,165万元，占营业收入约9.7%。\n2. 拥有发明专利31项、实用新型99项、外观设计6项。",
            "question": "研发投入及专利情况？",
        },
        {
            "block_index": 1,
            "total_pieces": 2,
            "pieces_content": "1. 募集资金用于补充流动资金及偿债。\n2. 截至报告期末已使用约60%。",
            "question": "募集资金用途及使用进度？",
        },
    ],

    "block_selection_structured": [
        {
            "block_index": 1,
            "total_pieces": 3,
            "pieces_content": EXAMPLE_PIECES_CONTENT,
            "question": EXAMPLE_QUESTION_RISK,
        },
        {
            "block_index": 1,
            "total_pieces": 3,
            "pieces_content": EXAMPLE_PIECES_CONTENT,
            "question": EXAMPLE_QUESTION_BASIC,
        },
        {
            "block_index": 2,
            "total_pieces": 2,
            "pieces_content": "1. 公司注册地为广东省东莞市厚街镇恒通路10号，法定代表人张三。\n2. 经营范围为软件开发与技术服务。",
            "question": "公司注册地址和法定代表人？",
        },
        {
            "block_index": 1,
            "total_pieces": 2,
            "pieces_content": "1. 研发费用75,165万元，占营业收入约9.7%。\n2. 发明专利31项、实用新型99项、外观设计6项，多项已应用。",
            "question": "研发费用及专利分布？",
        },
        {
            "block_index": 1,
            "total_pieces": 2,
            "pieces_content": "1. 募集资金拟用于补流及偿债。\n2. 截至报告期末已使用约60%，剩余按计划投入。",
            "question": "募集资金用途与已使用比例？",
        },
    ],
}

# ====== 运行所有模板 + 用例，对比两模型输出与用时 ======
ALL_TEMPLATE_RESULTS = []

for tmpl_name, cases in TEMPLATE_TEST_CASES.items():
    # content_generation_system / content_generation_human 在下方以组合形式单独测试，这里跳过
    if tmpl_name in ("content_generation_system", "content_generation_human"):
        continue

    print("\n" + "#" * 70)
    print(f"### 模板: {tmpl_name}  | 用例数: {len(cases)}")

    for idx, state in enumerate(cases, start=1):
        print("\n" + "=" * 60)
        print(f"【模板 {tmpl_name} | 用例 {idx}/{len(cases)}】")
        # 为避免打印过长，只展示部分 state
        try:
            preview_state = {k: (v[:80] + "...") if isinstance(v, str) and len(v) > 80 else v for k, v in state.items()}
        except Exception:
            preview_state = state
        print("state 预览:")
        pprint(preview_state)

        full_prompt = apply_prompt_template(tmpl_name, state)

        t0 = time.perf_counter()
        resp1 = model1.invoke(full_prompt)
        elapsed1 = time.perf_counter() - t0
        out1 = (resp1.content if hasattr(resp1, "content") else str(resp1)).strip()

        t0 = time.perf_counter()
        resp2 = model2.invoke(full_prompt)
        elapsed2 = time.perf_counter() - t0
        out2 = (resp2.content if hasattr(resp2, "content") else str(resp2)).strip()

        faster = MODEL_ID_1 if elapsed1 < elapsed2 else MODEL_ID_2

        ALL_TEMPLATE_RESULTS.append({
            "template": tmpl_name,
            "case_idx": idx,
            "state": state,
            "out1": out1,
            "out2": out2,
            "elapsed1": elapsed1,
            "elapsed2": elapsed2,
            "faster": faster,
        })

        print(f"  --- {MODEL_ID_1} [用时 {elapsed1:.2f}s] ---")
        print(f"      {out1}")
        print(f"  --- {MODEL_ID_2} [用时 {elapsed2:.2f}s] ---")
        print(f"      {out2}")
        faster_time = elapsed1 if faster == MODEL_ID_1 else elapsed2
        slower_time = elapsed2 if faster == MODEL_ID_1 else elapsed1
        ratio = faster_time / slower_time if slower_time > 0 else 1.0
        print(f"  更快: {faster}（{faster_time:.2f}/{slower_time:.2f}={ratio*100:.0f}%）")

# ====== content_generation_system + content_generation_human 组合测试 ======
sys_cases = TEMPLATE_TEST_CASES.get("content_generation_system", [])
human_cases = TEMPLATE_TEST_CASES.get("content_generation_human", [])
combined_cases = []

if human_cases:
    for i, h in enumerate(human_cases):
        simple = False
        if sys_cases:
            simple = bool(sys_cases[i % len(sys_cases)].get("simple_answer", False))
        case = {
            "simple_answer": simple,
            "question": h.get("question"),
            "context_text": h.get("context_text", ""),
            "extra_instruction": h.get("extra_instruction", ""),
        }
        combined_cases.append(case)

    print("\n" + "#" * 70)
    print(f"### 联合模板: content_generation_system + content_generation_human  | 用例数: {len(combined_cases)}")

    for idx, case in enumerate(combined_cases, start=1):
        print("\n" + "=" * 60)
        print(f"【联合模板 content_generation_system+human | 用例 {idx}/{len(combined_cases)}】")
        pprint(case)

        system_prompt = apply_prompt_template(
            "content_generation_system",
            {"simple_answer": case["simple_answer"]},
        )
        human_prompt = apply_prompt_template(
            "content_generation_human",
            {
                "question": case["question"],
                "context_text": case["context_text"],
                "extra_instruction": case["extra_instruction"],
            },
        )

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_prompt),
        ]

        t0 = time.perf_counter()
        resp1 = model1.invoke(messages)
        elapsed1 = time.perf_counter() - t0
        out1 = (resp1.content if hasattr(resp1, "content") else str(resp1)).strip()

        t0 = time.perf_counter()
        resp2 = model2.invoke(messages)
        elapsed2 = time.perf_counter() - t0
        out2 = (resp2.content if hasattr(resp2, "content") else str(resp2)).strip()

        faster = MODEL_ID_1 if elapsed1 < elapsed2 else MODEL_ID_2

        ALL_TEMPLATE_RESULTS.append({
            "template": "content_generation_system+human",
            "case_idx": idx,
            "state": case,
            "out1": out1,
            "out2": out2,
            "elapsed1": elapsed1,
            "elapsed2": elapsed2,
            "faster": faster,
        })

        print(f"  --- {MODEL_ID_1} [用时 {elapsed1:.2f}s] ---")
        print(f"      {out1}")
        print(f"  --- {MODEL_ID_2} [用时 {elapsed2:.2f}s] ---")
        print(f"      {out2}")
        faster_time = elapsed1 if faster == MODEL_ID_1 else elapsed2
        slower_time = elapsed2 if faster == MODEL_ID_1 else elapsed1
        ratio = faster_time / slower_time if slower_time > 0 else 1.0
        print(f"  更快: {faster}（{faster_time:.2f}/{slower_time:.2f}={ratio*100:.0f}%）")

print("\n" + "#" * 70)
print("所有模板测试完成，结果保存在 ALL_TEMPLATE_RESULTS 中（可在 notebook 中进一步分析 / 过滤）。")


2026-02-25 17:20:36.577 | INFO     | src.utils.model_factory:create_langchain_model:54 - Using default base_url[http://103.154.31.78:20001/compatible-mode/v1] and api_key[****] from environment variables for model model_bailian_80b



######################################################################
### 模板: retrieval_query_generation  | 用例数: 5

【模板 retrieval_query_generation | 用例 1/5】
state 预览:
{'description': '申请人的基础设施建设业务有哪些？基础设施板块中PPP项目的数量、投资与回款情况如何？回报机制以何种方式为主？是否存在回款周期长和运营不确定性风险？',
 'prompt': '请聚焦最近三年的数据，仅统计已完工项目；若无则说明暂无。'}
  --- model_bailian_80b [用时 1.30s] ---
      近三年已完工基础设施PPP项目数量、投资、回款情况及回报机制，是否存在回款周期长和运营不确定性风险
  --- model_4b_instruct [用时 0.26s] ---
      近三年已完工基础设施PPP项目投资回款及回报机制风险
  更快: model_4b_instruct（0.26/1.30=20%）

【模板 retrieval_query_generation | 用例 2/5】
state 预览:
{'description': '授信初审报告中的审查人是谁？', 'prompt': ''}
  --- model_bailian_80b [用时 0.72s] ---
      授信初审报告中的审查人是谁
  --- model_4b_instruct [用时 0.20s] ---
      授信初审报告的审查人是谁
  更快: model_4b_instruct（0.20/0.72=28%）

【模板 retrieval_query_generation | 用例 3/5】
state 预览:
{'description': '分行初审意见及联系方式', 'prompt': '需包含初审结论与联系人电话。'}
  --- model_bailian_80b [用时 0.70s] ---
      分行初审意见及联系人电话
  --- model_4b_instruct [用时 0.20s] ---
      分行初审意见及联系人电话
  更快: 

: 

In [ ]:
"""长 prompt 测试：对 storage/test_prompts/prompt{i}.md（i=1,2,...,PROMPT_NUM）依次读取已渲染的完整 prompt，对两模型对比输出与用时。

文件中直接为渲染好的 prompt 文本，无需关联任何 template。
"""

from pathlib import Path

# ========== 超参数：要测试的长 prompt 数量 ==========
PROMPT_NUM = 10

config_path = os.path.join(PROJECT_ROOT, "config", "model_factory_config.json")
factory = ChatModelFactory(config_path=config_path)
model1 = factory.create_langchain_model(model_id=MODEL_ID_1)
model2 = factory.create_langchain_model(model_id=MODEL_ID_2)

prompts_dir = Path(PROJECT_ROOT) / "storage" / "test_prompts"

for i in range(1, PROMPT_NUM + 1):
    prompt_path = prompts_dir / f"prompt{i}.md"
    print("\n" + "=" * 60)
    print(f"【长 prompt 测试 {i}/{PROMPT_NUM}】{prompt_path.name}")
    print("=" * 60)

    with open(prompt_path, "r", encoding="utf-8") as f:
        long_prompt = f.read()

    print(f"  字符数: {len(long_prompt)}，前 200 字: {long_prompt[:200]}...")
    print()

    print(f"  ------ 模型1: {MODEL_ID_1} ------")
    t0 = time.perf_counter()
    resp1 = model1.invoke(long_prompt)
    elapsed1 = time.perf_counter() - t0
    out1 = (resp1.content if hasattr(resp1, "content") else str(resp1)).strip()
    print(out1)
    print(f"  [用时] {elapsed1:.2f} 秒")

    print()
    print(f"  ------ 模型2: {MODEL_ID_2} ------")
    t0 = time.perf_counter()
    resp2 = model2.invoke(long_prompt)
    elapsed2 = time.perf_counter() - t0
    out2 = (resp2.content if hasattr(resp2, "content") else str(resp2)).strip()
    print(out2)
    print(f"  [用时] {elapsed2:.2f} 秒")

    faster = MODEL_ID_1 if elapsed1 < elapsed2 else MODEL_ID_2
    faster_time = elapsed1 if faster == MODEL_ID_1 else elapsed2
    slower_time = elapsed2 if faster == MODEL_ID_1 else elapsed1
    ratio = faster_time / slower_time if slower_time > 0 else 1.0
    print()
    print(f"  ====== 用时对比 ====== 更快: {faster}（{faster_time:.2f}/{slower_time:.2f}={ratio*100:.0f}%）")

print("\n" + "=" * 60)
print(f"长 prompt 测试完成，共 {PROMPT_NUM} 个文件。")


2026-02-26 16:05:51.646 | INFO     | src.utils.model_factory:create_langchain_model:54 - Using default base_url[http://103.154.31.78:20001/compatible-mode/v1] and api_key[****] from environment variables for model model_bailian_80b



【长 prompt 测试 1/1】prompt1.md
  字符数: 40020，前 200 字: 你是一位分析“分行初审意见及联系方式”问题与给定知识内容关系的专家助手。

# 工作要求：
- 评估现有知识内容是否足以直接回答用户问题
- 识别是否需要进一步计算处理或数据分析来解决该问题
- 如果现有内容足够回答问题，不需要生成计算框架
- 如果问题涉及数值计算、数据分析或复杂处理，且目前已经得到的计算结果不足以回答问题，则生成相应的计算处理框架

# 具体要求：
- 确保计算指令包含完整的处...

  ------ 模型1: model_bailian_80b ------
{
    "needs_computation": false,
    "reasoning": "问题询问的是‘分行初审意见及联系方式’，其中‘联系方式’可直接从文档中提取，如doc_5、doc_6、doc_9至doc_12、doc_15、doc_29等均明确列出了董事会秘书、联席公司秘书、证券事务代表的姓名、地址、电话、传真和电子邮箱，信息完整且一致。而‘分行初审意见’在doc_8和doc_30中明确提及‘北京银行脱敏B分行’作为调查机构，其尽职调查报告即为初审意见的载体，内容已完整呈现，无需进一步计算或处理。",
    "computation_prompt": "",
    "computation_input": []
}
  [用时] 4.16 秒

  ------ 模型2: model_4b_instruct ------
```json
{
    "needs_computation": false,
    "reasoning": "问题询问的是‘分行初审意见及联系方式’，在现有知识内容中，文档id为doc_8、doc_12、doc_15、doc_20、doc_21、doc_22、doc_25、doc_27、doc_30、doc_31、doc_32、doc_33、doc_34、doc_35、doc_36、doc_37、doc_38、doc_39、doc_40、doc_41、doc_42、doc_43、doc_44、doc_45、doc_46、doc_47等中，明确提到了北京银行脱敏B分行对企业的尽职调查报告，其中包含‘初审意见’相关内容，如授信额度、业务品